# Modelos — T033 (Multilayer Perceptron / Spark MLlib)

Notebook **rede neural** da M03 via **`MultilayerPerceptronClassifier`** (Spark MLlib), alinhado ao protocolo **T030** (`temperature_C`, split 70/30, `seed=42`).

## Limitacao da API Spark

O Spark 3.x expoe rede feedforward só como **classificacao**. Para manter **MAE / RMSE / R² em graus Celsius** (comparável ao [`decision_tree.ipynb`](decision_tree.ipynb)), o alvo contínuo é **discretizado em quantis** (`QuantileDiscretizer`, apenas **fit no treino**). As classes previstas são mapeadas ao **ponto médio do intervalo** do bin (erro de quantização controlado com `NUM_BUCKETS`).

As entradas passam por **`StandardScaler` (z-score, fit no treino)** — como no [`neural_network_numpy_training.ipynb`](neural_network_numpy_training.ipynb). **Sem escalar**, o MLP costuma **colapsar para uma única classe** (predição decodificada quase constante em °C).

## Objetivo (T033)

- Documentar camadas (`layers`), iteracoes (`maxIter`), `blockSize`, solver.
- Funcao de perda **interna ao Spark**: softmax + entropia cruzada sobre classes dos bins; métricas finais são **regressão em °C** após descodificar predição.
- Tempo de treino + hardware; métricas **treino vs teste** como no T031.

### Cluster Standalone / HDFS

- Por defeito no `docker-compose`, `SPARK_MASTER` pode ser `local[4]` no serviço `notebook`. Para usar **workers** (`spark-worker`), defina `JUPYTER_SPARK_MASTER=spark://spark-master:7077` e ajuste memória dos executores.
- Opcional: ler Parquet em **HDFS** (`hdfs://namenode:9000/...`) depois de copiar o ficheiro para o cluster; caso contrário mantém-se `/dataset/Indian_Weather_Dataset.parquet` montado.

### Epocas / historico de treino (vs notebook NumPy)

- No NumPy (`neural_network_numpy_training.ipynb`) o loop **epoch-a-epoch** calcula MAE/RMSE/R² no **validação**.
- No Spark MLlib o L-BFGS otimiza **sem API estável** para MAE em °C por passo; usamos **`mlp_model.summary.objectiveHistory`** (objectivo multinomial / regularização) como curva de optimização — **não** é a mesma métrica que R² no VAL do notebook antigo.
- Os splits do `QuantileDiscretizer` vêm com **`-inf` / `+inf`** nos extremos; sem correção, o ponto médio do primeiro/último bin vira **NaN** e o `RegressionEvaluator` devolve **inf**.

- Leituras de **`show(N)`**: sem **`orderBy`**, o Spark mostra uma fatia não ordenada do plano (partições) — várias linhas seguidas podem ter **a mesma predição**, mesmo quando existem **muitas classes distintas**. As células abaixo usam amostragem **`orderBy(rand(seed))`** e extremos por **|erro|**.

- Comparar ao [`neural_network_numpy_training.ipynb`](neural_network_numpy_training.ipynb): lá o split é **temporal** (T021); aqui mantém **`randomSplit` 70/30** como o [`decision_tree.ipynb`](decision_tree.ipynb). Os números e o comportamento não serão paralelos apenas por ser dois 'MLP'.


In [ ]:
from pathlib import Path
import glob
import os
import platform
import shutil
import socket
import sys
import time

# nbconvert / subprocess sem kernel Jupyter: SPARK_HOME/python + py4j zip na lib/.
_spark_home = os.environ.get("SPARK_HOME") or "/usr/local/spark"
_spark_py = os.path.join(_spark_home, "python")
if os.path.isdir(_spark_py) and _spark_py not in sys.path:
    sys.path.insert(0, _spark_py)
_py4j_candidates = sorted(glob.glob(os.path.join(_spark_home, "python", "lib", "py4j-*-src.zip")))
if _py4j_candidates:
    _py4j_zip = _py4j_candidates[-1]
    if _py4j_zip not in sys.path:
        sys.path.insert(0, _py4j_zip)

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, lit, udf
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType
from pyspark.storagelevel import StorageLevel
from pyspark.ml.feature import QuantileDiscretizer, StandardScaler, VectorAssembler,Bucketizer
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import RegressionEvaluator

TARGET_COL = "temperature_C"
SEED = 42
TRAIN_RATIO = 0.7

NUM_BUCKETS = 32
QD_RELATIVE_ERROR = 0.001

SAMPLE_FRACTION = 1
MAX_ROWS = None
_dtr_rows = os.environ.get("DTR_NOTEBOOK_MAX_ROWS", "").strip()
if _dtr_rows and MAX_ROWS is None:
    MAX_ROWS = int(_dtr_rows)

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

candidate_parquets = [
    Path("/dataset/Indian_Weather_Dataset.parquet"),
    REPO_ROOT / "data" / "Indian_Weather_Dataset.parquet",
]
PARQUET_PATH = next((p for p in candidate_parquets if p.exists()), None)
if PARQUET_PATH is None:
    raise FileNotFoundError(
        "Parquet nao encontrado. Confirme que existe em data/ ou que ./data foi montado em /dataset no docker-compose."
    )

if not os.environ.get("JAVA_HOME"):
    ms = Path(r"C:\\Program Files\\Microsoft")
    if ms.is_dir():
        for jdk in sorted(ms.glob("jdk-*-hotspot"), reverse=True):
            if (jdk / "bin" / "java.exe").is_file():
                os.environ["JAVA_HOME"] = str(jdk)
                break
    if not os.environ.get("JAVA_HOME"):
        java_exe = shutil.which("java")
        if java_exe:
            jp = Path(java_exe).resolve()
            if jp.parent.name.lower() == "bin":
                os.environ["JAVA_HOME"] = str(jp.parent.parent)
if not os.environ.get("JAVA_HOME"):
    raise RuntimeError(
        "Defina JAVA_HOME para um JDK 17+ (ex.: Microsoft OpenJDK). No VS Code: Settings > Python > Env File ou env do kernel."
    )

_spark_master_raw = os.environ.get("SPARK_MASTER")
spark_master = (_spark_master_raw or "").strip() or "local[4]"
if not spark_master.lower().startswith("local") and spark_master.startswith("spark://"):
    try:
        hostport = spark_master[len("spark://") :].split("/", 1)[0]
        host = hostport.rsplit(":", 1)[0]
        socket.gethostbyname(host)
    except OSError:
        print(f"[aviso] SPARK_MASTER={spark_master!r} nao resolve aqui; usando local[4].")
        spark_master = "local[4]"


def _reset_spark_if_stale() -> None:
    try:
        inst = getattr(SparkSession, "_instantiatedSession", None)
        if inst is not None:
            sc = getattr(inst, "_sc", None)
            if sc is not None:
                try:
                    sc.stop()
                except Exception:
                    pass
    except Exception:
        pass
    try:
        SparkSession._instantiatedSession = None
        SparkSession._activeSession = None
    except Exception:
        pass
    try:
        from pyspark.sql.context import SQLContext

        SQLContext._instantiatedContext = None
    except Exception:
        pass
    SparkContext._gateway = None
    SparkContext._jvm = None
    SparkContext._active_spark_context = None


_reset_spark_if_stale()
try:
    spark.stop()
except Exception:
    pass
_reset_spark_if_stale()

_is_local_master = str(spark_master).lower().startswith("local")
_driver_mem = os.environ.get("SPARK_DRIVER_MEMORY") or (
    "8g" if _is_local_master else "4g"
)
_shuffle_parts = os.environ.get("SPARK_SQL_SHUFFLE_PARTITIONS", "16")
_default_par = os.environ.get("SPARK_DEFAULT_PARALLELISM", "8")

builder = (
    SparkSession.builder.appName("T033_MultilayerPerceptronClassifier")
    .master(spark_master)
    .config("spark.driver.memory", _driver_mem)
    .config("spark.sql.shuffle.partitions", _shuffle_parts)
    .config("spark.default.parallelism", _default_par)
)
if _is_local_master:
    _max_result = os.environ.get("SPARK_DRIVER_MAX_RESULT_SIZE", "2g")
    builder = builder.config("spark.driver.maxResultSize", _max_result)
if not str(spark_master).lower().startswith("local"):
    driver_host = os.environ.get("SPARK_DRIVER_HOST", "spark-notebook")
    _exec_mem = os.environ.get("SPARK_EXECUTOR_MEMORY", "1g")
    _exec_cores = os.environ.get("SPARK_EXECUTOR_CORES", "1")
    _cores_max = os.environ.get("SPARK_CORES_MAX", "4")
    builder = (
        builder.config("spark.driver.host", driver_host)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.executor.memory", _exec_mem)
        .config("spark.executor.cores", _exec_cores)
        .config("spark.cores.max", _cores_max)
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.network.timeout", "600s")
        .config("spark.executor.heartbeatInterval", "120s")
    )

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("REPO_ROOT:", REPO_ROOT)
print("PARQUET_PATH:", PARQUET_PATH)
print("SPARK_MASTER:", spark_master)
print("spark.sparkContext.master:", spark.sparkContext.master)
print("spark.version:", spark.version)
print("spark.driver.memory (pedido):", _driver_mem)
if not str(spark_master).lower().startswith("local"):
    print("spark.driver.host:", driver_host)
    print(
        "spark.executor (mem/cores/max):",
        os.environ.get("SPARK_EXECUTOR_MEMORY", "1g"),
        os.environ.get("SPARK_EXECUTOR_CORES", "1"),
        os.environ.get("SPARK_CORES_MAX", "4"),
    )
print("SAMPLE_FRACTION:", SAMPLE_FRACTION)
print("MAX_ROWS:", MAX_ROWS)
print("NUM_BUCKETS (classes / ultima camada):", NUM_BUCKETS)


REPO_ROOT: /home/jovyan/work
PARQUET_PATH: /dataset/Indian_Weather_Dataset.parquet
SPARK_MASTER: local[4]
spark.sparkContext.master: local[4]
spark.version: 3.2.1
spark.driver.memory (pedido): 3g
SAMPLE_FRACTION: 1
MAX_ROWS: 800000
NUM_BUCKETS (classes / ultima camada): 64


In [2]:
try:
    df = spark.read.parquet(str(PARQUET_PATH))
except Exception as exc:
    msg = str(exc)
    if "getSubject is not supported" in msg:
        raise RuntimeError(
            "Falha do Spark/Hadoop com Java atual (getSubject). "
            "Use Java 17 para o processo do Jupyter e reinicie o kernel."
        ) from exc
    raise

numeric_types = (DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType)
exclude_cols = {TARGET_COL, "rain_label"}
feature_cols = [
    f.name
    for f in df.schema.fields
    if isinstance(f.dataType, numeric_types) and f.name not in exclude_cols
]

if not feature_cols:
    raise ValueError("Nenhuma feature numerica encontrada para o treino.")

model_df = df.select([TARGET_COL] + feature_cols).na.drop(subset=[TARGET_COL])
model_df = model_df.fillna(0.0, subset=feature_cols)

if SAMPLE_FRACTION < 1.0:
    model_df = model_df.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)

if MAX_ROWS is not None and MAX_ROWS > 0:
    model_df = model_df.limit(MAX_ROWS)

model_df = model_df.coalesce(8)
model_df = model_df.persist(StorageLevel.DISK_ONLY)

print("N features:", len(feature_cols))
print("Features usadas:", feature_cols)
print("Preview dos dados:")
model_df.limit(5).show(truncate=False)


N features: 17
Features usadas: ['lat', 'lon', 'humidity_pct', 'pressure_hPa', 'dew_point_C', 'pressure_trend', 'solar_radiation_Wm2', 'wind_speed_ms', 'cloud_cover_pct', 'hour', 'month', 'wind_direction_deg', 'wind_dir_sin', 'wind_dir_cos', 'cape', 'et0_mm', 'precip_mm']
Preview dos dados:
+-------------+-------+------+------------+------------+-----------+--------------+-------------------+-------------+---------------+----+-----+------------------+------------------+-------------------+----+------+---------+
|temperature_C|lat    |lon   |humidity_pct|pressure_hPa|dew_point_C|pressure_trend|solar_radiation_Wm2|wind_speed_ms|cloud_cover_pct|hour|month|wind_direction_deg|wind_dir_sin      |wind_dir_cos       |cape|et0_mm|precip_mm|
+-------------+-------+------+------------+------------+-----------+--------------+-------------------+-------------+---------------+----+-----+------------------+------------------+-------------------+----+------+---------+
|20.6         |19.6641|78.532|64 

In [ ]:
import math

from pyspark.sql.functions import max as F_max, min as F_min

train_df, test_df = model_df.randomSplit([TRAIN_RATIO, 1.0 - TRAIN_RATIO], seed=SEED)

qd = QuantileDiscretizer(
    numBuckets=NUM_BUCKETS,
    inputCol=TARGET_COL,
    outputCol="labelIndex",
    relativeError=QD_RELATIVE_ERROR,
)
qd_model = qd.fit(train_df)
train_bin = qd_model.transform(train_df)
test_bin = qd_model.transform(test_df)


def qd_edges(model):
    sp = model.getSplits()
    if not sp or len(sp) == 0:
        raise ValueError("QuantileDiscretizer sem splits")
    if isinstance(sp[0], (list, tuple)):
        inner = sp[0]
    else:
        inner = sp
    return [float(x) for x in inner]


def sanitize_quantile_splits(edge_list, lo, hi):
    """Spark usa -inf/+inf nos extremos; medias com inf geram NaN e metricas inf no evaluator."""
    out = []
    for x in edge_list:
        xf = float(x)
        if math.isnan(xf):
            raise ValueError("split NaN no QuantileDiscretizer")
        if math.isinf(xf) and xf < 0:
            out.append(lo)
        elif math.isinf(xf) and xf > 0:
            out.append(hi)
        else:
            out.append(xf)
    return out


_bounds = train_df.select(F_min(TARGET_COL).alias("lo"), F_max(TARGET_COL).alias("hi")).first()
_lo, _hi = float(_bounds["lo"]), float(_bounds["hi"])

edges_raw = qd_edges(qd_model)
edges = sanitize_quantile_splits(edges_raw, _lo, _hi)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

train_full = assembler.transform(train_bin)
test_full = assembler.transform(test_bin)

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withMean=True, withStd=True)
scaler_model = scaler.fit(train_full)
_tr_s = scaler_model.transform(train_full)
_te_s = scaler_model.transform(test_full)
train_full = _tr_s.drop("features").withColumnRenamed("scaledFeatures", "features")
test_full = _te_s.drop("features").withColumnRenamed("scaledFeatures", "features")
print("StandardScaler aplicado (fit=só treino); coluna `features` do MLP = vetor normalizado.")

train_fit = train_full.select(
    col("labelIndex").cast("double").alias("label"),
    "features",
)

_input_dim = int(train_fit.select("features").first()[0].size)
layers = [_input_dim, 128, 64, NUM_BUCKETS]

print("Arquitetura layers (entrada, ocultas..., classes):", layers)
print("Primeiros splits (apos sanitizar -inf/+inf com min/max treino):", edges[:5], "... total", len(edges))
print("Preview treino (features + indice de classe):")
train_fit.show(3, truncate=False)


StandardScaler aplicado (fit=só treino); coluna `features` do MLP = vetor normalizado.
Arquitetura layers (entrada, ocultas..., classes): [17, 64, 32, 64]
Primeiros splits (apos sanitizar -inf/+inf com min/max treino): [2.2, 10.2, 12.3, 13.8, 14.9] ... total 65
Preview treino (features + indice de classe):
+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|features                                                                                                                                                                                                                                                                                                                   |
+-----+-------------------------------------

In [ ]:
params_mlp = {
    "layers": layers,
    "maxIter": 300,
    "blockSize": 128,
    "solver": "l-bfgs",
    "tol": 1e-6,
    "seed": SEED,
}

mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=params_mlp["maxIter"],
    blockSize=params_mlp["blockSize"],
    solver=params_mlp["solver"],
    tol=params_mlp["tol"],
    seed=params_mlp["seed"],
    layers=params_mlp["layers"],
)

print("Hiperparametros MultilayerPerceptronClassifier:")
for k, v in params_mlp.items():
    print(f"  - {k}: {v}")

t0 = time.perf_counter()
mlp_model = mlp.fit(train_fit)
train_secs = time.perf_counter() - t0

print("Tempo de treino (s):", round(train_secs, 2))
print("Hardware:", platform.machine(), platform.system(), platform.processor() or "n/d")
print("Modelo treinado.")

# Historico L-BFGS (substituto das "epocas" visiveis no loop NumPy — aqui e objectivo interno, nao MAE em °C).
import matplotlib.pyplot as plt

objective_history = []
if getattr(mlp_model, "hasSummary", False) and mlp_model.hasSummary:
    try:
        objective_history = [float(x) for x in mlp_model.summary.objectiveHistory]
    except Exception as _exc:
        print("Aviso: objectiveHistory indisponivel:", _exc)

print("historico objectivo L-BFGS (len ~ iteracoes+1):", len(objective_history))
if len(objective_history) > 1:
    print("  inicial:", objective_history[0], "| final:", objective_history[-1])
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.plot(range(len(objective_history)), objective_history, marker=".", linestyle="-")
    ax.set_xlabel("Indice (0 = antes da 1ª iteracao; depois cada passo L-BFGS)")
    ax.set_ylabel("Objectivo (multiclasse / regularizacao Spark)")
    ax.set_title("T033 MLP Spark — objectiveHistory (nao e R2 nem MAE em °C)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("(Sem curva: summary vazio ou API sem objectiveHistory.)")


Hiperparametros MultilayerPerceptronClassifier:
  - layers: [17, 64, 32, 64]
  - maxIter: 100
  - blockSize: 128
  - solver: l-bfgs
  - tol: 1e-06
  - seed: 42
Tempo de treino (s): 265.37
Hardware: x86_64 Linux x86_64
Modelo treinado.
Aviso: objectiveHistory indisponivel: 'function' object has no attribute 'objectiveHistory'
historico objectivo L-BFGS (len ~ iteracoes+1): 0
(Sem curva: summary vazio ou API sem objectiveHistory.)


In [5]:
from pyspark.sql.functions import abs as F_abs, desc, isnan, max as spark_max, min as spark_min, rand

n_bins = len(edges) - 1
if n_bins != NUM_BUCKETS:
    print(f"[aviso] len(edges)-1={n_bins} != NUM_BUCKETS={NUM_BUCKETS} (ajuste fino se necessario)")

bc_edges = spark.sparkContext.broadcast(edges)


@udf(DoubleType())
def decode_class_to_temp(pred):
    import math

    e = bc_edges.value
    niv = len(e) - 1  # numero de intervalos / classes
    if pred is None:
        return None
    v = float(pred)
    if math.isnan(v):
        return None
    i = int(v)
    if i < 0:
        i = 0
    if i >= niv:
        i = niv - 1
    return float((e[i] + e[i + 1]) / 2.0)


def eval_reg(pred_df):
    clean = pred_df.filter(col("label").isNotNull() & col("prediction").isNotNull()).filter(
        ~isnan(col("label")) & ~isnan(col("prediction"))
    )
    mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae").evaluate(clean)
    rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse").evaluate(clean)
    r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2").evaluate(clean)
    return {"MAE": float(mae), "RMSE": float(rmse), "R2": float(r2)}


def scored_to_cont(pred_cls_df):
    return pred_cls_df.select(
        col(TARGET_COL).alias("label"),
        decode_class_to_temp(col("prediction")).alias("prediction"),
    )


inp_train = train_full.withColumn("label", col("labelIndex").cast("double")).drop("labelIndex")
pred_train_cls = mlp_model.transform(inp_train)

inp_test = test_full.withColumn("label", col("labelIndex").cast("double")).drop("labelIndex")
pred_test_cls = mlp_model.transform(inp_test)

_dist_tr = pred_train_cls.select("prediction").distinct().count()
_dist_te = pred_test_cls.select("prediction").distinct().count()
print("Indices de classe distintos previstos — treino:", _dist_tr, "| teste:", _dist_te)
if _dist_te <= 2:
    print(
        "[aviso] Predicao quase constante em classes; conferir scaler, maxIter e dados. "
        "Com scaler + dados OK deve haver dezenas de classes distintas.")

pred_mlp_train = scored_to_cont(pred_train_cls)
pred_mlp_test = scored_to_cont(pred_test_cls)

_rg = pred_mlp_test.select(
    spark_min("label").alias("y_min"),
    spark_max("label").alias("y_max"),
    spark_min("prediction").alias("pred_min"),
    spark_max("prediction").alias("pred_max"),
).first()
print(
    "Faixa no teste (°C) — label real:", float(_rg["y_min"]), "…", float(_rg["y_max"]),
    "| predicao decodificada:", float(_rg["pred_min"]), "…", float(_rg["pred_max"]),
)

_pv = pred_mlp_test.withColumn("abs_err", F_abs(col("label") - col("prediction")))
print("Amostra aleatoria reproducivel (ordenar antes de olhar):")
_pv.select("label", "prediction", "abs_err").orderBy(rand(SEED)).limit(10).show(truncate=False)
print("Piores linhas por |erro| no teste:")
_pv.orderBy(desc("abs_err")).select("label", "prediction", "abs_err").limit(5).show(truncate=False)
print("Linhas com menor |erro| no teste:")
_pv.orderBy("abs_err").select("label", "prediction", "abs_err").limit(5).show(truncate=False)

baseline_value = train_df.select(avg(TARGET_COL).alias("avg_label")).first()["avg_label"]
pred_base = (
    test_df.select(TARGET_COL).withColumnRenamed(TARGET_COL, "label").withColumn("prediction", lit(float(baseline_value)))
)

m_base = eval_reg(pred_base)
m_mlp_test = eval_reg(pred_mlp_test)
m_mlp_train = eval_reg(pred_mlp_train)

bc_edges.unpersist()

rows_teste = [
    ("baseline_media_global", m_base["MAE"], m_base["RMSE"], m_base["R2"]),
    ("mlp_multiclass_discretizado_t033_degC", m_mlp_test["MAE"], m_mlp_test["RMSE"], m_mlp_test["R2"]),
]

rows_gap = [
    ("MAE", m_mlp_train["MAE"], m_mlp_test["MAE"], m_mlp_test["MAE"] - m_mlp_train["MAE"]),
    ("RMSE", m_mlp_train["RMSE"], m_mlp_test["RMSE"], m_mlp_test["RMSE"] - m_mlp_train["RMSE"]),
    ("R2", m_mlp_train["R2"], m_mlp_test["R2"], m_mlp_train["R2"] - m_mlp_test["R2"]),
]

print("Metricas (teste):")
print(f"{'modelo':<36} {'MAE':>10} {'RMSE':>10} {'R2':>10}")
print("-" * 68)
for modelo, mae, rmse, r2 in rows_teste:
    print(f"{modelo:<36} {mae:>10.4f} {rmse:>10.4f} {r2:>10.4f}")
print("-" * 68)

print("Multilayer Perceptron (descodificado): treino vs teste")
print(f"{'split':<10} {'MAE':>10} {'RMSE':>10} {'R2':>10}")
print("-" * 44)
print(f"{'treino':<10} {m_mlp_train['MAE']:>10.4f} {m_mlp_train['RMSE']:>10.4f} {m_mlp_train['R2']:>10.4f}")
print(f"{'teste':<10} {m_mlp_test['MAE']:>10.4f} {m_mlp_test['RMSE']:>10.4f} {m_mlp_test['R2']:>10.4f}")
print("-" * 44)

print("Gap de generalizacao (teste - treino para erro; treino - teste para R2):")
for nome, tr, te, gap in rows_gap:
    print(f"- {nome}: treino={tr:.4f} | teste={te:.4f} | gap={gap:+.4f}")


Indices de classe distintos previstos — treino: 61 | teste: 61
Faixa no teste (°C) — label real: 2.5 … 46.8 | predicao decodificada: 6.199999999999999 … 43.6
Amostra aleatoria reproducivel (ordenar antes de olhar):
+-----+------------------+-------------------+
|label|prediction        |abs_err            |
+-----+------------------+-------------------+
|25.6 |25.6              |0.0                |
|26.6 |26.6              |0.0                |
|26.0 |26.299999999999997|0.29999999999999716|
|23.5 |23.0              |0.5                |
|35.8 |34.7              |1.0999999999999943 |
|26.4 |26.299999999999997|0.10000000000000142|
|23.0 |23.0              |0.0                |
|28.6 |28.950000000000003|0.3500000000000014 |
|26.4 |26.299999999999997|0.10000000000000142|
|23.6 |23.65             |0.04999999999999716|
+-----+------------------+-------------------+

Piores linhas por |erro| no teste:
+-----+----------+------------------+
|label|prediction|abs_err           |
+-----+--------

In [ ]:
import matplotlib.pyplot as plt

# 1. Extrair o histórico do sumário do modelo
# O L-BFGS salva o valor da função objetivo a cada iteração
if mlp_model.hasSummary:
    objective_history = mlp_model.summary.objectiveHistory
    
    # 2. Criar o gráfico
    plt.figure(figsize=(10, 5))
    plt.plot(objective_history, sorted=False, color='#4C72B0', linewidth=2)
    
    plt.title('Curva de Convergência (L-BFGS)', fontsize=14)
    plt.xlabel('Iteração', fontsize=12)
    plt.ylabel('Função Objetivo (Loss)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # Adicionar anotação do valor final
    plt.annotate(f'Loss final: {objective_history[-1]:.4f}', 
                 xy=(len(objective_history)-1, objective_history[-1]),
                 xytext=(len(objective_history)*0.7, objective_history[0]*0.8),
                 arrowprops=dict(facecolor='black', shrink=0.05))
    
    plt.show()
else:
    print("O modelo não gerou um sumário. Verifique se o solver é l-bfgs.")

In [6]:
MODEL_DIR = REPO_ROOT / "models" / "multilayer_perceptron_t033"
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

try:
    mlp_model.write().overwrite().save(str(MODEL_DIR))
    print("Modelo salvo em:", MODEL_DIR)
except Exception as exc:
    print("Aviso: nao foi possivel salvar o modelo Spark neste ambiente local.")
    print("Motivo:", str(exc)[:300])
    print("Treino e metricas seguem validos; apenas o artefato Spark nao foi persistido.")

spark.stop()
print("Spark finalizado.")


Modelo salvo em: /home/jovyan/work/models/multilayer_perceptron_t033
Spark finalizado.
